# Accent Coach — Calibration Recording Studio

**Run each cell top-to-bottom once, then use the widget in the last cell.**

- Click **⏺ Record** → say the sentence → click **⏹ Stop**
- Click **▶ Play back** to hear yourself
- Click **✓ Save & Next** (or **↺ Re-record** to redo)
- Recordings land in `tts_output/accent_coach/users/owner/`

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve()))

import threading
import numpy as np
import sounddevice as sd
import soundfile as sf
import ipywidgets as widgets
from IPython.display import display, Audio, clear_output

from accent_coach.calibration.sentences import CALIBRATION_SENTENCES

OUT_DIR = Path("../tts_output/accent_coach/users/owner")
OUT_DIR.mkdir(parents=True, exist_ok=True)
SR = 44100  # standard mic rate; parselmouth handles any SR

def wav_path(sentence) -> Path:
    slug = sentence.text[:30].lower()
    slug = "".join(c if c.isalnum() else "_" for c in slug).strip("_")
    return OUT_DIR / f"{sentence.id:03d}_{slug}.wav"

already_done = [s for s in CALIBRATION_SENTENCES if wav_path(s).exists()]
print(f"Output dir : {OUT_DIR.resolve()}")
print(f"Already recorded: {len(already_done)} / {len(CALIBRATION_SENTENCES)}")


Output dir : /Users/ivkrasovskii/model-voice-generator/tts_output/accent_coach/users/owner
Already recorded: 0 / 50


In [ ]:
class RecordingStudio:
    def __init__(self):
        self._buf: list[np.ndarray] = []
        self._stream: sd.InputStream | None = None
        self._recording = False
        self._current_audio: np.ndarray | None = None

        # Find first unrecorded sentence
        self._idx = 0
        for i, s in enumerate(CALIBRATION_SENTENCES):
            if not wav_path(s).exists():
                self._idx = i
                break

        # ── Widgets ──────────────────────────────────────────────────
        self._progress = widgets.HTML()
        self._sentence_box = widgets.HTML()
        self._target_box = widgets.HTML()
        self._status = widgets.HTML(value="<i>Ready.</i>")

        self._btn_record = widgets.Button(
            description="⏺ Record", button_style="danger",
            layout=widgets.Layout(width="140px", height="40px")
        )
        self._btn_stop = widgets.Button(
            description="⏹ Stop", button_style="warning",
            disabled=True, layout=widgets.Layout(width="140px", height="40px")
        )
        self._btn_play = widgets.Button(
            description="▶ Play back", button_style="info",
            disabled=True, layout=widgets.Layout(width="140px", height="40px")
        )
        self._btn_save = widgets.Button(
            description="✓ Save & Next", button_style="success",
            disabled=True, layout=widgets.Layout(width="140px", height="40px")
        )
        self._btn_redo = widgets.Button(
            description="↺ Re-record", button_style="",
            disabled=True, layout=widgets.Layout(width="140px", height="40px")
        )
        self._btn_prev = widgets.Button(
            description="← Back", layout=widgets.Layout(width="100px", height="40px")
        )
        self._btn_skip = widgets.Button(
            description="Skip →", layout=widgets.Layout(width="100px", height="40px")
        )
        self._audio_out = widgets.Output()

        self._btn_record.on_click(self._on_record)
        self._btn_stop.on_click(self._on_stop)
        self._btn_play.on_click(self._on_play)
        self._btn_save.on_click(self._on_save)
        self._btn_redo.on_click(self._on_redo)
        self._btn_prev.on_click(lambda _: self._jump(-1))
        self._btn_skip.on_click(lambda _: self._jump(+1))

        self._layout = widgets.VBox([
            self._progress,
            self._sentence_box,
            self._target_box,
            widgets.HBox([self._btn_record, self._btn_stop]),
            widgets.HBox([self._btn_play, self._btn_save, self._btn_redo]),
            widgets.HBox([self._btn_prev, self._btn_skip]),
            self._status,
            self._audio_out,
        ])
        self._refresh_ui()

    # ── Recording ────────────────────────────────────────────────────

    def _on_record(self, _):
        self._buf = []
        self._current_audio = None
        self._recording = True
        self._stream = sd.InputStream(samplerate=SR, channels=1, dtype="float32",
                                       callback=self._audio_cb)
        self._stream.start()
        self._btn_record.disabled = True
        self._btn_stop.disabled = False
        self._btn_play.disabled = True
        self._btn_save.disabled = True
        self._btn_redo.disabled = True
        self._status.value = "<b style='color:red'>● Recording…</b>"

    def _audio_cb(self, indata, frames, time, status):
        if self._recording:
            self._buf.append(indata.copy())

    def _on_stop(self, _):
        self._recording = False
        if self._stream:
            self._stream.stop()
            self._stream.close()
            self._stream = None
        if self._buf:
            self._current_audio = np.concatenate(self._buf, axis=0).flatten()
        self._btn_record.disabled = False
        self._btn_stop.disabled = True
        self._btn_play.disabled = self._current_audio is None
        self._btn_save.disabled = self._current_audio is None
        self._btn_redo.disabled = self._current_audio is None
        dur = len(self._current_audio) / SR if self._current_audio is not None else 0
        self._status.value = f"<i>Stopped. {dur:.1f}s captured.</i>"

    def _on_play(self, _):
        if self._current_audio is None:
            return
        self._audio_out.clear_output()
        with self._audio_out:
            display(Audio(data=self._current_audio, rate=SR, autoplay=True))

    def _on_save(self, _):
        if self._current_audio is None:
            return
        s = CALIBRATION_SENTENCES[self._idx]
        p = wav_path(s)
        sf.write(str(p), self._current_audio, SR)
        self._status.value = f"<b style='color:green'>Saved → {p.name}</b>"
        self._audio_out.clear_output()
        self._jump(+1)

    def _on_redo(self, _):
        self._current_audio = None
        self._btn_play.disabled = True
        self._btn_save.disabled = True
        self._btn_redo.disabled = True
        self._status.value = "<i>Cleared. Record again.</i>"
        self._audio_out.clear_output()

    def _jump(self, delta: int):
        new_idx = self._idx + delta
        if 0 <= new_idx < len(CALIBRATION_SENTENCES):
            self._idx = new_idx
        self._current_audio = None
        self._btn_play.disabled = True
        self._btn_save.disabled = True
        self._btn_redo.disabled = True
        self._audio_out.clear_output()
        self._refresh_ui()

    # ── UI ───────────────────────────────────────────────────────────

    def _refresh_ui(self):
        done = sum(1 for s in CALIBRATION_SENTENCES if wav_path(s).exists())
        pct = done / len(CALIBRATION_SENTENCES) * 100
        self._progress.value = (
            f"<b>Progress: {done} / {len(CALIBRATION_SENTENCES)}</b> "
            f"<span style='color:gray'>({pct:.0f}%)</span>"
        )

        s = CALIBRATION_SENTENCES[self._idx]
        exists = wav_path(s).exists()
        badge = " <span style='color:green'>✓ recorded</span>" if exists else ""
        self._sentence_box.value = (
            f"<div style='font-size:1.6em; padding:12px 0; line-height:1.4;'>"
            f"<b>#{s.id}/50</b>{badge}<br>"
            f"<span style='font-size:1.2em'>\"{s.text}\"</span>"
            f"</div>"
        )
        self._target_box.value = (
            f"<span style='color:#888; font-size:0.9em'>"
            f"Focus: {', '.join(s.targets)} &nbsp;|&nbsp; Type: {s.sentence_type}"
            f"</span>"
        )
        if not self._recording:
            self._status.value = (
                "<b style='color:green'>Already recorded — re-record or skip.</b>"
                if exists else "<i>Ready. Click Record when you're set.</i>"
            )

    def show(self):
        display(self._layout)


studio = RecordingStudio()
studio.show()
